In [ ]:
from quspin.operators import hamiltonian # 用于在给定的基（basis）上构建哈密顿量算符(或其他物理观测量)；
from quspin.basis import spin_basis_1d # 用于创建一维自旋-1/2链的希尔伯特空间基；
import numpy as np 
from numpy import linalg as LA
import matplotlib.pyplot as plt  # 用于结果可视化
from ncon import ncon
from One_dimensional_Model import Heisenberg_Model
from One_dimensional_Model import Ising_Model
from One_dimensional_Model import Tight_Binding_Model
from One_dimensional_Model import Hubbard_Model
from One_dimensional_Model import tJ_Model

import sys
sys.path.append(r'D:\Jupyter\DMRG') # sys.path：Python在导入模块时搜索的路径列表
from DMRG_Loc import DMRG_two_site

In [ ]:
########################## t-J模型取半满填充情形(即空穴率为0，此时t-J模型退化到海森堡模型)的测试 #################################
#### 参数设置
L = 16 # 格点数；
hole_doping = 0 # 空穴率，即空穴数与格点数的比值；
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.05 # 相互作用系数；
t2 = 0
J2 = 0
boundary='periodic' # 边界条件(默认周期边界条件periodic)；

#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('-------------------------ED计算结果--------------------------')
print()

#### 哈密顿量的构建
H, basis = tJ_Model(L, hole_doping, t1, J1, t2, J2, boundary)

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

## 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
print()
print(f"系统基态能量: {E_gs:.15f}")
print(f"每格点能量: {E_gs/L:.15f}")



#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('-------------------------海森堡模型计算结果--------------------------')
print()

#### 哈密顿量的构建
H, basis = Heisenberg_Model(L, J1, J2, boundary)

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

## 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
E_gs = E_gs - 1/4 * J1 * L
print()
print(f"系统基态能量: {E_gs:.15f}")
print(f"每格点能量: {E_gs/L:.15f}")

In [ ]:
#################################### Hubbard模型取U=0情形(即有自旋的Tight binding模型)的测试 ######################################

#### 参数设置
L = 12 # 格点数；
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
Nf = (L//2,L//2) # 填充粒子数，形式为元组Nf=(N_up,N_down),其中N_up上自旋粒子数,N_down下自旋粒子数
boundary='open'

#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('-------------------------ED计算结果--------------------------')
print()

#### 哈密顿量的构建
U = 0 # 相互作用系数；
t2=0
H, basis = Hubbard_Model(L, t1, U, Nf, t2, boundary)

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

## 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)
print('='*80)
print('系统能量：')
print()
print(f"计算基态能量: {E_gs:.15f}")
print(f"每格点能量: {E_gs/L:.15f}")


#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('------------------------解析计算结果--------------------------')
print()
## 开放边界条件OBC情形：
H_OBC = np.diag(np.ones(L-1),k=1) + np.diag(np.ones(L-1),k=-1) # 费米子的度量矩阵为左右次对角线上全为1，其余全为0的Nsites*Nsites矩阵
# np.diag(A,k=1)指将数组A的值依次放在右次对角线上(由k=1决定)；同理，k=-1便是放在左次对角线上，k=2便是放在次次对角线上……
D_OBC = LA.eigvalsh(H_OBC) # eigvalsh专用于计算厄密矩阵或实对称矩阵的特征值，其计算效率高，输出特征值数组默认按升序排列。
EnExact_OBC = 2 * sum(D_OBC[D_OBC < 0]) # 筛选出所有负特征值(操作D[D < 0])并对其求和(操作sum),最后乘2
print(f"OBC下基态能量: {EnExact_OBC:.15f}")
print(f"OBC下单格点能量: {EnExact_OBC/L:.15f}")
print()

## 周期边界条件PBC情形：
# 构建费米子度量矩阵
H_PBC = np.diag(np.ones(L-1), k=1) + np.diag(np.ones(L-1), k=-1) # 此处与OBC相同
# 添加周期边界条件的角落元素
H_PBC[0, L-1] = 1 # 矩阵最右上端元素；
H_PBC[L-1, 0] = 1 # 矩阵最左下端元素；
D_PBC = LA.eigvalsh(H_PBC)
EnExact_PBC = 2 * sum(D_PBC[D_PBC < 0])
print(f"PBC下基态能量: {EnExact_PBC:.15f}")
print(f"PBC下单格点能量: {EnExact_PBC/L:.15f}")
print()



In [ ]:
#######################################海森堡模型的测试######################################

# 共用参数：
L = 16 # 格点数
J = 1.0 # 海森堡耦合强度
boundary='open'

#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
############严格对角化(Exact Diagonalization)的计算
print('='*80)
print('------------------------ED计算结果--------------------------')
print()
    
##### 一、定义基(包括各种对称性的考虑)
basis = spin_basis_1d(L=L, pauli=False, Nup=None,   
    kblock=None, # 特别注意：开放边界条件下kblock=None(可以不写表示默认为None)；
    pblock=None, 
    zblock=None,
    a=1         
)

##### 二、哈密顿量的构建：
### 2.1构建耦合列表。
bond_list_xy = [[J/2, i, (i+1)%L] for i in range(L)] # 自旋x、y对应的耦合列表(将Sx与Sy用S+与S-表示则矩阵都是实矩阵,但会多出现1/2)；
bond_list_zz = [[J, i, (i+1)%L] for i in range(L)] # 自旋z对应的耦合列表

### 2.2构建哈密顿量
static = [["+-", bond_list_xy], ["-+", bond_list_xy]] # "+-"表示S+S-
dynamic = [] 
H = hamiltonian(static, dynamic, basis=basis, dtype=np.float64, check_symm=False) 

### 2.3查看返回的H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状
print()

#### 三、物理量计算验证
# 计算基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA')
print(f"基态能量: {E_gs[0]:.15f}")
print(f"单格点能量: {E_gs[0]/L:.15f}")
print()
    

#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------    
############ 严格计算结果[与精确结果对比(通过自由费米子方法计算得到)]

print('='*80)
print('------------------------解析计算结果--------------------------')
print()
#### 对一维海森堡XX模型，通过Jordan-Wigner变换得到自由费米子的紧束缚模型来解析求解(XXX模型则不行)
## 开放边界条件OBC情形：
H_OBC = np.diag(np.ones(L-1),k=1) + np.diag(np.ones(L-1),k=-1) # 费米子的度量矩阵为左右次对角线上全为1，其余全为0的Nsites*Nsites矩阵
# np.diag(A,k=1)指将数组A的值依次放在右次对角线上(由k=1决定)；同理，k=-1便是放在左次对角线上，k=2便是放在次次对角线上……
D_OBC = LA.eigvalsh(H_OBC) # eigvalsh专用于计算厄密矩阵或实对称矩阵的特征值，其计算效率高，输出特征值数组默认按升序排列。
EnExact_OBC = 0.5*sum(D_OBC[D_OBC < 0]) # 筛选出所有负特征值(操作D[D < 0])并对其求和(操作sum),最后乘2
print(f"OBC下XY模型基态能量: {EnExact_OBC:.15f}")
print(f"OBC下XY模型单格点能量: {EnExact_OBC/L:.15f}")
print()

## 周期边界条件PBC情形：
# 构建费米子度量矩阵
H_PBC = np.diag(np.ones(L-1), k=1) + np.diag(np.ones(L-1), k=-1) # 此处与OBC相同
# 添加周期边界条件的角落元素
H_PBC[0, L-1] = -1 # 矩阵最右上端元素；
H_PBC[L-1, 0] = -1 # 矩阵最左下端元素；
D_PBC = LA.eigvalsh(H_PBC)
EnExact_PBC = 0.5 * sum(D_PBC[D_PBC < 0])
print(f"PBC下XY模型基态能量: {EnExact_PBC:.15f}")
print(f"PBC下XY模型单格点能量: {EnExact_PBC/L:.15f}")
print()

#### XXX模型的Bethe ansatz解析计算结果：
E0_infinite = (1/4 - np.log(2)) # 约 -0.443147
print(f"海森堡模型热力学极限每格点能量: {E0_infinite:.15f}")



#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
############密度矩阵重整化群(Density Matrix Renormalization Group)的计算

print('='*80)
print('------------------------DMRG计算结果--------------------------')
#### 一、定义系统的物理指标维数chid,并创建储存基态能量的列表
chid = 2
Energy = []

#### 二、DMRG_Loc的输入参数
numsweeps = 4 # number of DMRG sweeps
dispon = 0 # level of output display(打印模式)
updateon = True # level of output display(是否进行待更新张量的优化)
maxit = 10 # iterations of Lanczos method(Lanczos算法的迭代次数)
krydim = 5 # dimension of Krylov subspace(Krylov子空间的维度)

#### 三、定义局域哈密顿量(Define Local Hamiltonian)：Heisenberg XXX model
Sx = np.array([[0,1],[1,0]])
Sy = np.array([[0,-1j],[1j,0]])
Sz = np.array([[1, 0], [0,-1]])
Su = np.array([[0, 1], [0, 0]])
Sd = np.array([[0, 0], [1, 0]])
I = np.eye(chid)
h = (np.real(np.kron(Sx,Sx) + np.kron(Sy,Sy))).reshape(2,2,2,2)
hLs = np.zeros((2,2)).reshape(1,2,1,2) # 开放左边界条件
hRs = np.zeros((2,2)).reshape(2,1,2,1) # 开放右边界条件

for chi in range(100, 110, 10):
    #### 四、创建储存初始OBC-MPS的列表A(生成随机均匀分布的局域张量)s
    A = [0 for x in range(L)]
    A[0] = np.random.rand(1,chid,min(chid,chi)) 
    for p in range(1,L):
        chil = A[p-1].shape[2]
        A[p] = np.random.rand(chil, chid, min(chil*chid,chid**(L-1-p),chi))
        
    #### 五、DMRG计算基态能量
    En, A, sWeight, B = DMRG_two_site(A, hLs, h, hRs, chi, numsweeps, dispon, updateon, maxit, krydim)
    print(f"基态能量: {En[-1]/4:.15f}, Bond dim: {chi}")
    print(f"单格点能量: {En[-1]/(4*L):.15f}, Bond dim: {chi}")
    Energy.append(En[-1])

In [ ]:
#################################### Tight binding模型(即无自旋的Hubbard模型取U=0情形)的测试 ######################################

#### 参数设置
L = 20 # 格点数；
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
t2 = 0
Nf = L//2 # 填充粒子数，形式为元组Nf=(N_up,N_down),其中N_up上自旋粒子数,N_down下自旋粒子数
boundary='periodic'

#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('-------------------------ED计算结果--------------------------')
print()
#### 哈密顿量的构建
U = 0 # 相互作用系数；
t2=0
H, basis = Tight_Binding_Model(L, t1, t2, boundary, Nf)

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

## 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)
print('='*80)
print('系统能量：')
print()
print(f"计算基态能量: {E_gs:.15f}")
print(f"每格点能量: {E_gs/L:.15f}")


#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
print('='*80)
print('------------------------解析计算结果--------------------------')
print()
## 开放边界条件OBC情形：
H_OBC = np.diag(np.ones(L-1),k=1) + np.diag(np.ones(L-1),k=-1) # 费米子的度量矩阵为左右次对角线上全为1，其余全为0的Nsites*Nsites矩阵
# np.diag(A,k=1)指将数组A的值依次放在右次对角线上(由k=1决定)；同理，k=-1便是放在左次对角线上，k=2便是放在次次对角线上……
D_OBC = LA.eigvalsh(H_OBC) # eigvalsh专用于计算厄密矩阵或实对称矩阵的特征值，其计算效率高，输出特征值数组默认按升序排列。
EnExact_OBC = sum(D_OBC[D_OBC < 0]) # 筛选出所有负特征值(操作D[D < 0])并对其求和(操作sum),最后乘2
print(f"OBC下基态能量: {EnExact_OBC:.15f}")
print(f"OBC下单格点能量: {EnExact_OBC/L:.15f}")
print()

## 周期边界条件PBC情形：
# 构建费米子度量矩阵
H_PBC = np.diag(np.ones(L-1), k=1) + np.diag(np.ones(L-1), k=-1) # 此处与OBC相同
# 添加周期边界条件的角落元素
H_PBC[0, L-1] = 1 # 矩阵最右上端元素；
H_PBC[L-1, 0] = 1 # 矩阵最左下端元素；
D_PBC = LA.eigvalsh(H_PBC)
EnExact_PBC = sum(D_PBC[D_PBC < 0])
print(f"PBC下基态能量: {EnExact_PBC:.15f}")
print(f"PBC下单格点能量: {EnExact_PBC/L:.15f}")
print()

